# Data Quality Audit — 04 · DataSaudi Tourism Indicators (11 files)

**Source:** `data/raw/tourism_statistics/tourism_statistics_datasaudi/*.csv`

**Purpose in the concierge:** Official statistics: purpose of visit, sentiment (NPS/satisfaction), occupancy, macro indicators.

Standardized audit covering:

```
Dataset
├── Shape
├── Columns & data types
├── Missing values
├── Duplicates
├── Invalid values
├── Outliers
├── Inconsistent categories
├── Geographic validity
├── Date/time validity
├── Data-source/license
└── Known limitations
```

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

# Resolve repo root whether run from the audit folder or the repo root.
p = Path.cwd()
while p != p.parent and not (p / "data" / "raw").exists():
    p = p.parent
ROOT = p
print("repo root:", ROOT)

# Saudi Arabia bounding box (approx) for geographic validity checks.
SA_LAT = (16.0, 32.5)
SA_LON = (34.5, 56.0)

def iqr_outliers(series):
    """Return (count, lower, upper) of IQR outliers in a numeric series."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return 0, np.nan, np.nan
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lo) | (s > hi)).sum()), round(lo, 2), round(hi, 2)

def missing_report(df):
    m = pd.DataFrame({"missing": df.isna().sum(),
                      "missing_%": (df.isna().mean() * 100).round(1)})
    return m[m["missing"] > 0].sort_values("missing", ascending=False)

def dtype_report(df):
    return pd.DataFrame({
        "dtype": [str(t) for t in df.dtypes],
        "non_null": df.notna().sum().values,
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    }, index=df.columns)


repo root: /home/user/saudi-Digital-Concierge


In [2]:
DIR = ROOT / "data/raw/tourism_statistics/tourism_statistics_datasaudi"
files = sorted(DIR.glob("*.csv"))
data = {f.name: pd.read_csv(f) for f in files}
print(f"Loaded {len(data)} DataSaudi files")

Loaded 11 DataSaudi files


## Shape (per file)

In [3]:
shape = pd.DataFrame([{"file": n.replace(".csv",""), "rows": len(d), "cols": d.shape[1]} for n, d in data.items()])
print("Total rows:", shape["rows"].sum())
shape

Total rows: 1975


,file,rows,cols
0,mot_average_length_of_stay_by_tourists_type,49,8
1,mot_average_spending_per_night_by_tourists_type,49,8
2,mot_average_spending_per_trip_by_tourists_type,49,8
3,mot_number_of_overnight_stays_by_tourists_type,49,8
4,mot_tourism_by_trip_main_purpose_and_tourists_...,127,8
5,net_promoter_score,6,5
6,sama_tourism_expenditure_by_trip_purpose_and_t...,126,8
7,tourism_complementary_indicators,6,24
8,tourism_occupancy_rate_monthly,1392,8
9,tourism_occupancy_rate_yearly,116,6


## Columns & data types (per file)

In [4]:
for n, d in data.items():
    print(f"\n=== {n} ===")
    print(list(d.columns))


=== mot_average_length_of_stay_by_tourists_type.csv ===
['Year', 'Economic Sectors ID', 'Economic Sectors', 'Nationality ID', 'Nationality', 'Tourist Type Name ID', 'Tourist Type Name', 'Night']

=== mot_average_spending_per_night_by_tourists_type.csv ===
['Year', 'Economic Sectors ID', 'Economic Sectors', 'Nationality ID', 'Nationality', 'Tourist Type Name ID', 'Tourist Type Name', 'SAR']

=== mot_average_spending_per_trip_by_tourists_type.csv ===
['Year', 'Economic Sectors ID', 'Economic Sectors', 'Nationality ID', 'Nationality', 'Tourist Type Name ID', 'Tourist Type Name', 'SAR']

=== mot_number_of_overnight_stays_by_tourists_type.csv ===
['Year', 'Economic Sectors ID', 'Economic Sectors', 'Nationality ID', 'Nationality', 'Tourist Type Name ID', 'Tourist Type Name', 'Overnight Stays']

=== mot_tourism_by_trip_main_purpose_and_tourists_type.csv ===
['Year', 'Economic Sectors ID', 'Economic Sectors', 'Nationality ID', 'Nationality', 'Trip Purpose Name ID', 'Trip Purpose Name', 'Touri

## Missing values & Duplicates (per file)

In [5]:
rows = []
for n, d in data.items():
    rows.append({"file": n.replace(".csv",""),
                 "missing": int(d.isna().sum().sum()),
                 "duplicates": int(d.duplicated().sum())})
pd.DataFrame(rows)

,file,missing,duplicates
0,mot_average_length_of_stay_by_tourists_type,0,0
1,mot_average_spending_per_night_by_tourists_type,0,0
2,mot_average_spending_per_trip_by_tourists_type,0,0
3,mot_number_of_overnight_stays_by_tourists_type,0,0
4,mot_tourism_by_trip_main_purpose_and_tourists_...,0,0
5,net_promoter_score,0,0
6,sama_tourism_expenditure_by_trip_purpose_and_t...,0,0
7,tourism_complementary_indicators,0,0
8,tourism_occupancy_rate_monthly,0,0
9,tourism_occupancy_rate_yearly,0,0


## Invalid values
Ratio-type indicators (satisfaction, occupancy) should sit in [0, 1]; NPS in [-100, 100].

In [6]:
def rng(name, col, lo, hi):
    for n, d in data.items():
        if col in d.columns:
            s = pd.to_numeric(d[col], errors="coerce")
            print(f"{name}: {n} range {round(s.min(),3)}..{round(s.max(),3)} out_of[{lo},{hi}]={int(((s<lo)|(s>hi)).sum())}")
rng("Satisfaction", "Tourism Satisfaction Index", 0, 1)
rng("Occupancy(y)", "Occupancy Rate", 0, 1)
rng("NPS", "Net Promoter Score", -100, 100)

Satisfaction: tourism_satisfaction_index.csv range 0.6..0.65 out_of[0,1]=0
Occupancy(y): tourism_occupancy_rate_monthly.csv range 0.134..0.858 out_of[0,1]=0
Occupancy(y): tourism_occupancy_rate_yearly.csv range 0.234..0.722 out_of[0,1]=0
NPS: net_promoter_score.csv range 68.0..77.0 out_of[-100,100]=0


## Outliers
DataSaudi tables are aggregated official series — outliers reflect real spread, shown for the volume tables.

In [7]:
for n, d in data.items():
    numcols = [c for c in d.columns if d[c].dtype.kind in "fi" and not c.lower().endswith("id") and c != "Year"]
    for col in numcols[:1]:
        k, lo, hi = iqr_outliers(d[col])
        print(f"{n[:45]:45s} [{col[:22]:22s}] outliers={k}")

mot_average_length_of_stay_by_tourists_type.c [Night                 ] outliers=0
mot_average_spending_per_night_by_tourists_ty [SAR                   ] outliers=0
mot_average_spending_per_trip_by_tourists_typ [SAR                   ] outliers=0
mot_number_of_overnight_stays_by_tourists_typ [Overnight Stays       ] outliers=2
mot_tourism_by_trip_main_purpose_and_tourists [Tourists              ] outliers=6
net_promoter_score.csv                        [Net Promoter Score    ] outliers=0
sama_tourism_expenditure_by_trip_purpose_and_ [Million SAR           ] outliers=7
tourism_complementary_indicators.csv          [Gross Travel Propensit] outliers=0
tourism_occupancy_rate_monthly.csv            [Occupancy Rate        ] outliers=13
tourism_occupancy_rate_yearly.csv             [Occupancy Rate        ] outliers=0
tourism_satisfaction_index.csv                [Tourism Satisfaction I] outliers=0


## Inconsistent categories
Dimension vocabularies across files (tourist type, nationality, purpose, accommodation, province).

In [8]:
for dim in ["Tourist Type Name","Nationality","Trip Purpose Name",
            "Accommodation Type","Province"]:
    vals = set()
    for d in data.values():
        if dim in d.columns:
            vals |= set(d[dim].dropna().unique())
    if vals:
        print(f"{dim}: {sorted(vals)}")

Tourist Type Name: ['Domestic', 'Inbound', 'Outbound']
Nationality: ['Non Saudi', 'Saudi', 'Total']
Trip Purpose Name: ['Business', 'Leisure', 'Other', 'Religious', 'Visiting Friends & Relatives']
Accommodation Type: ['Apartments and Others', 'Grand Total', 'Hotels']
Province: ['Al-Baha', 'Al-Jouf', 'Al-Madinah Al-Monawarah', 'Al-Qaseem', 'Al-Riyadh', 'Aseer', 'Eastern Region', 'Grand Total', 'Hail', 'Jazan', 'Makkah Al-Mokarramah', 'Najran', 'Northern Borders', 'Tabouk']


## Geographic validity
Only the occupancy tables carry `Province` (no coordinates). Note `Grand Total` rollup rows and Saudi-specific spellings.

In [9]:
occ = data.get("tourism_occupancy_rate_yearly.csv")
if occ is not None:
    print("Provinces:", sorted(occ["Province"].unique()))
    print("\nGrand Total rollup rows present:", int((occ["Province"]=="Grand Total").sum()))

Provinces: ['Al-Baha', 'Al-Jouf', 'Al-Madinah Al-Monawarah', 'Al-Qaseem', 'Al-Riyadh', 'Aseer', 'Eastern Region', 'Grand Total', 'Hail', 'Jazan', 'Makkah Al-Mokarramah', 'Najran', 'Northern Borders', 'Tabouk']

Grand Total rollup rows present: 12


## Date/time validity
Year ranges vary by file; sentiment tables use `Quarter`.

In [10]:
for n, d in data.items():
    if "Year" in d.columns:
        yr = d["Year"]
        print(f"{n[:50]:50s} Year {int(yr.min())}-{int(yr.max())}")
    elif "Quarter" in d.columns:
        q = d["Quarter"]
        print(f"{n[:50]:50s} Quarter {q.min()}..{q.max()}")

mot_average_length_of_stay_by_tourists_type.csv    Year 2015-2025
mot_average_spending_per_night_by_tourists_type.cs Year 2015-2025
mot_average_spending_per_trip_by_tourists_type.csv Year 2015-2025
mot_number_of_overnight_stays_by_tourists_type.csv Year 2015-2025
mot_tourism_by_trip_main_purpose_and_tourists_type Year 2015-2025
net_promoter_score.csv                             Quarter 2024-Q2..2025-Q3
sama_tourism_expenditure_by_trip_purpose_and_touri Year 2014-2022
tourism_complementary_indicators.csv               Year 2019-2024
tourism_occupancy_rate_monthly.csv                 Year 2021-2024
tourism_occupancy_rate_yearly.csv                  Year 2021-2024
tourism_satisfaction_index.csv                     Quarter 2024-Q2..2025-Q3


## Data-source / license
- **Source:** DataSaudi (official government open-data portal).
- **License:** Official / open government data.
- **Currency:** periodically updated (2015–2025); treat as *depends* — re-pull for latest.

## Known limitations
- **Redundant columns**: `Economic Sectors` constant, plus paired `*_ID` columns.
- **`Grand Total` rollup rows** must be excluded before aggregation.
- **Province spellings** (`Al-Riyadh`, `Makkah Al-Mokarramah`) differ from other sources.
- Several files **duplicate metrics** already in Source 3; monthly occupancy (1,392 rows) is heavy for RAG.
- Mixed year coverage across files.